In [ ]:
%env MUJOCO_GL=egl
import cv2
import torch
import numpy as np
from scipy.spatial.transform import Rotation
import mediapy

import mujoco
from gaussian_renderer import GSRendererMuJoCo

from gs_playground import ROOT_PATH
from gs_playground.src.manipulation.robots.universal_robots_ur5e_robotiq.ur5e_robotiq import UR5eRobotiq

_ASSETS_UR5E_DIR = ROOT_PATH / "models" / "robots" / "manipulation" / "universal_robots_ur5e_robotiq"
mjcf_path = _ASSETS_UR5E_DIR / "xmls/table30_02_stack_color_blocks.xml"

gaussians = UR5eRobotiq.robot_gaussians()
gaussians["background"] = UR5eRobotiq.robot_background_ply()

## Single Env

In [ ]:
# Make model, data, and renderer
mj_model = mujoco.MjModel.from_xml_path(mjcf_path.as_posix())
mj_data = mujoco.MjData(mj_model)
mujoco.mj_resetDataKeyframe(mj_model, mj_data, 0)
mujoco.mj_forward(mj_model, mj_data)

H = 240
W = 320
renderer = mujoco.Renderer(mj_model, H, W)
renderer.update_scene(mj_data, 0)
img = renderer.render()

gsmj_renderer = GSRendererMuJoCo(gaussians, mj_model)
gsmj_renderer.update_gaussians(mj_data)
results = gsmj_renderer.render(mj_model, mj_data, list(range(mj_model.ncam)), W, H)
rgb = (255 * torch.clamp(results[0][0], 0., 1.)).to(torch.uint8).cpu().numpy()

mixed = cv2.addWeighted(img.astype(np.float32), 0.5, rgb.astype(np.float32), 0.5, 0).astype(np.uint8)

mediapy.show_image(np.hstack([img, mixed, rgb]))
